In [0]:
%run ./NB_00_config_loader.py

[SecretScope(name=' kv-insclm-cap-11'), SecretScope(name='kv-insclm')]

[SecretMetadata(key='adls-abfss-base'),
 SecretMetadata(key='adls-account-key'),
 SecretMetadata(key='adls-account-name'),
 SecretMetadata(key='adls-audit-path'),
 SecretMetadata(key='adls-base-url'),
 SecretMetadata(key='adls-bronze-path'),
 SecretMetadata(key='adls-container-name'),
 SecretMetadata(key='adls-gold-path'),
 SecretMetadata(key='adls-raw-path'),
 SecretMetadata(key='adls-rejected-path'),
 SecretMetadata(key='adls-silver-path'),
 SecretMetadata(key='database-workspace-url'),
 SecretMetadata(key='databricks-cluster-id'),
 SecretMetadata(key='databricks-pat'),
 SecretMetadata(key='file-claim-status-updates'),
 SecretMetadata(key='file-claims'),
 SecretMetadata(key='file-customer-master'),
 SecretMetadata(key='file-policy-master'),
 SecretMetadata(key='github-pat'),
 SecretMetadata(key='github-repo-url'),
 SecretMetadata(key='sql-admin-name'),
 SecretMetadata(key='sql-admin-password'),
 SecretMetadata(key='sql-connection-string'),
 SecretMetadata(key='sql-database-name'),
 S

✅ Config loaded from Key Vault successfully.
   ADLS Account  : [REDACTED]
   Container     : [REDACTED]
   ABFSS Base    : [REDACTED]
   RAW path      : [REDACTED][REDACTED]
   BRONZE path   : [REDACTED][REDACTED]
   SILVER path   : [REDACTED][REDACTED]
   GOLD path     : [REDACTED][REDACTED]
   REJECTED path : [REDACTED][REDACTED]
   AUDIT path    : [REDACTED][REDACTED]
   SQL Server    : [REDACTED]
   SQL Database  : [REDACTED]


In [0]:
# ── Cell 2 ───────────────────────────────────────────────────
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

dbutils.widgets.text("runId", "manual_run")
RUN_ID = dbutils.widgets.get("runId")

print(f"Run ID     : {RUN_ID}")
print(f"Start time : {datetime.now()}")

Run ID     : manual_run
Start time : 2026-05-21 14:20:30.309425


In [0]:
# ── Cell 3 ───────────────────────────────────────────────────
# Unity Catalog — NO LOCATION clause
spark.sql("CREATE DATABASE IF NOT EXISTS bronze_insclm")
print("✅ bronze_insclm database ready")

✅ bronze_insclm database ready


In [0]:
# ── Cell 4 ───────────────────────────────────────────────────
def ingest_to_bronze(source_path, target_table, schema):
    print(f"\n📥 Reading: {source_path}")
    df = (spark.read
            .option("header", "true")
            .option("dateFormat", "yyyy-MM-dd")
            .schema(schema)
            .csv(source_path)
            .withColumn("_ingestion_timestamp",
                current_timestamp())
            .withColumn("_source_file",
                F.col("_metadata.file_path"))
            .withColumn("_pipeline_run_id", lit(RUN_ID)))

    row_count = df.count()
    print(f"   Rows read : {row_count:,}")

    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table))

    saved = spark.table(target_table).count()
    print(f"✅ {target_table} → {saved:,} rows")
    return saved

In [0]:
# ── Cell 5 ───────────────────────────────────────────────────
claims_schema = StructType([
    StructField("claim_id",        StringType(),      False),
    StructField("policy_id",       StringType(),      False),
    StructField("customer_id",     StringType(),      False),
    StructField("claim_date",      DateType(),        True),
    StructField("claim_amount",    DecimalType(18,2), True),
    StructField("claim_reason",    StringType(),      True),
    StructField("document_status", StringType(),      True),
    StructField("ingestion_date",  DateType(),        True),
])

status_schema = StructType([
    StructField("status_update_id", StringType(), False),
    StructField("claim_id",         StringType(), False),
    StructField("old_status",       StringType(), True),
    StructField("new_status",       StringType(), True),
    StructField("status_date",      DateType(),   True),
    StructField("remarks",          StringType(), True),
])

policy_schema = StructType([
    StructField("policy_id",       StringType(),      False),
    StructField("customer_id",     StringType(),      False),
    StructField("policy_type",     StringType(),      True),
    StructField("coverage_amount", DecimalType(18,2), True),
    StructField("premium_amount",  DecimalType(18,2), True),
    StructField("policy_status",   StringType(),      True),
    StructField("start_date",      DateType(),        True),
    StructField("end_date",        DateType(),        True),
    StructField("last_updated",    TimestampType(),   True),
])

customer_schema = StructType([
    StructField("customer_id",   StringType(),    False),
    StructField("customer_name", StringType(),    True),
    StructField("email",         StringType(),    True),
    StructField("phone",         StringType(),    True),
    StructField("city",          StringType(),    True),
    StructField("state",         StringType(),    True),
    StructField("dob",           StringType(),      True),
    StructField("risk_category", StringType(),    True),
    StructField("last_updated",  TimestampType(), True),
])

print("✅ Schemas defined")


✅ Schemas defined


In [0]:
# ── Cell 6 ───────────────────────────────────────────────────
# Drop recursiveFileLookup from ingest function
# Use exact paths

c1 = ingest_to_bronze(
    RAW_PATH + "/claims/claims.csv",
    "bronze_insclm.bronze_claims",
    claims_schema
)

c2 = ingest_to_bronze(
    RAW_PATH + "/claim_status_updates/claim_status_updates.csv",
    "bronze_insclm.bronze_claim_status_updates",
    status_schema
)

# For policy and customer — use just ONE file from dated folder
c3 = ingest_to_bronze(
    RAW_PATH + "/policy_master/2026/05/21/policy_master_4a45d9d2-f377-48b2-8949-8895c5a976db.csv",
    "bronze_insclm.bronze_policy_master",
    policy_schema
)

c4 = ingest_to_bronze(
    RAW_PATH + "/customer_master/2026/05/21/customer_master_68432939-9842-4e9a-aec0-39604402f72b.csv",
    "bronze_insclm.bronze_customer_master",
    customer_schema
)


📥 Reading: [REDACTED][REDACTED]/claims/[REDACTED]
   Rows read : 2,200
✅ bronze_insclm.bronze_claims → 2,200 rows

📥 Reading: [REDACTED][REDACTED]/claim_status_updates/[REDACTED]
   Rows read : 1,600
✅ bronze_insclm.bronze_claim_status_updates → 1,600 rows

📥 Reading: [REDACTED][REDACTED]/policy_master/2026/05/21/policy_master_4a45d9d2-f377-48b2-8949-8895c5a976db.csv
   Rows read : 1,500
✅ bronze_insclm.bronze_policy_master → 1,500 rows

📥 Reading: [REDACTED][REDACTED]/customer_master/2026/05/21/customer_master_68432939-9842-4e9a-aec0-39604402f72b.csv
   Rows read : 1,000
✅ bronze_insclm.bronze_customer_master → 1,000 rows


In [0]:
# ── Cell 7 ───────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STARTING BRONZE INGESTION")
print("=" * 55)

start = datetime.now()

c1 = ingest_to_bronze(
    RAW_PATH + "/claims/claims.csv",
    "bronze_insclm.bronze_claims",
    claims_schema
)

c2 = ingest_to_bronze(
    RAW_PATH + "/claim_status_updates/claim_status_updates.csv",
    "bronze_insclm.bronze_claim_status_updates",
    status_schema
)

c3 = ingest_to_bronze(
    RAW_PATH + "/policy_master/2026/05/21/policy_master_4a45d9d2-f377-48b2-8949-8895c5a976db.csv",
    "bronze_insclm.bronze_policy_master",
    policy_schema
)

c4 = ingest_to_bronze(
    RAW_PATH + "/customer_master/2026/05/21/customer_master_68432939-9842-4e9a-aec0-39604402f72b.csv",
    "bronze_insclm.bronze_customer_master",
    customer_schema
)
print(f"bronze_customer_master : {c4:,}  (expected 1,000)")

end = datetime.now()

# ── Summary ───────────────────────────────────────────────────
print("\n" + "=" * 55)
print("BRONZE INGESTION SUMMARY")
print("=" * 55)
print(f"bronze_claims               : {c1:,}  (expected 2,200)")
print(f"bronze_claim_status_updates : {c2:,}  (expected 1,600)")
print(f"bronze_policy_master        : {c3:,}  (expected 1,500)")
print(f"bronze_customer_master      : {c4:,}  (expected 1,000)")
print(f"Total time                  : {end - start}")
print("=" * 55)

all_ok = (c1==2200 and c2==1600 and c3==1500 and c4==1000)
if all_ok:
    print("✅ All counts match. Bronze layer complete.")
    print("   Next: Run NB_02 Cells 1-4")
else:
    print("❌ Count mismatch — check paths above")


STARTING BRONZE INGESTION

📥 Reading: [REDACTED][REDACTED]/claims/[REDACTED]
   Rows read : 2,200
✅ bronze_insclm.bronze_claims → 2,200 rows

📥 Reading: [REDACTED][REDACTED]/claim_status_updates/[REDACTED]
   Rows read : 1,600
✅ bronze_insclm.bronze_claim_status_updates → 1,600 rows

📥 Reading: [REDACTED][REDACTED]/policy_master/2026/05/21/policy_master_4a45d9d2-f377-48b2-8949-8895c5a976db.csv
   Rows read : 1,500
✅ bronze_insclm.bronze_policy_master → 1,500 rows

📥 Reading: [REDACTED][REDACTED]/customer_master/2026/05/21/customer_master_68432939-9842-4e9a-aec0-39604402f72b.csv
   Rows read : 1,000
✅ bronze_insclm.bronze_customer_master → 1,000 rows
bronze_customer_master : 1,000  (expected 1,000)

BRONZE INGESTION SUMMARY
bronze_claims               : 2,200  (expected 2,200)
bronze_claim_status_updates : 1,600  (expected 1,600)
bronze_policy_master        : 1,500  (expected 1,500)
bronze_customer_master      : 1,000  (expected 1,000)
Total time                  : 0:00:15.652547
✅ All

In [0]:
# Keep only 1 file per folder — delete duplicates

# Claims — delete ADF files, keep manual upload
# (manual upload already correct at root level)
dbutils.fs.rm(RAW_PATH + "/claims/2026/", recurse=True)
print("✅ Duplicate claims files removed")

# Claim status updates — check first
files = dbutils.fs.ls(RAW_PATH + "/claim_status_updates/")
for f in files:
    print(f.path)

✅ Duplicate claims files removed
[REDACTED][REDACTED]/claim_status_updates/2026/
[REDACTED][REDACTED]/claim_status_updates/[REDACTED]


In [0]:
spark.sql("DROP TABLE IF EXISTS bronze_insclm.bronze_customer_master")
spark.sql("DROP TABLE IF EXISTS silver_insclm.silver_customer_dim")
print("✅ Dropped — ready to re-ingest")

✅ Dropped — ready to re-ingest
